# Modelo Regional y Nacional — Proyecciones 2026-2032

Proyecciones de producción de cobre por región y a nivel nacional (Chile).
Modelos: XGBoost y LightGBM con validación rolling-origin (2010–2018).

**Target**: `LogRatio = log(Prod_{origin+h} / Prod_origin)` — naive predice 0 (producción plana).

## Sección 1 — Imports & Configuración

In [1]:
import pandas as pd
import numpy as np
import warnings
import json
import os
import xgboost as xgb
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error
from scipy import stats
from scipy.stats import linregress

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

BASE = "/Users/mac/TrabajoTesis/FinalResultsFolder"
OUT  = f"{BASE}/03_Forecasting/regional_model/outputs"
os.makedirs(OUT, exist_ok=True)

RANDOM_STATE = 42
ORIGINS      = list(range(2010, 2019))   # 2010..2018
HORIZONS     = list(range(1, 8))          # H+1..H+7
N_TRIALS     = 50

# ---------------------------------------------------------------------------
# Mine → Region mapping
# ---------------------------------------------------------------------------
MINE_REGION = {
    # Region I - Tarapacá
    'collahuasi':                        'I',
    'quebrada blanca':                   'I',
    'cerro colorado':                    'I',
    # Region II - Antofagasta
    'escondida':                         'II',
    'chuquicamata':                      'II',
    'radomiro tomic':                    'II',
    'gabriela mistral':                  'II',
    'ministro hales':                    'II',
    'spence':                            'II',
    'centinela_centinela_sulfuros_':     'II',
    'centinela_centinela_óxidos_':       'II',
    'zaldivar':                          'II',
    'antucoya':                          'II',
    'michilla':                          'II',
    'lomas bayas':                       'II',
    'mantos blancos':                    'II',
    'franke':                            'II',
    'sierra gorda':                      'II',
    'el abra':                           'II',
    'capstone copper (4)':               'II',
    # Region III - Atacama
    'candelaria':                        'III',
    'caserones':                         'III',
    'salvador':                          'III',
    'mantoverde':                        'III',
    'atacama kozan':                     'III',
    'cerro negro':                       'III',
    'ojos del salado':                   'III',
    # Region IV - Coquimbo
    'andacollo':                         'IV',
    'los pelambres':                     'IV',
    'tres valles':                       'IV',
    'altos de punitaqui':                'IV',
    'haldeman':                          'IV',
    # Region V - Valparaíso
    'el soldado':                        'V',
    'andina':                            'V',
    # Region RM - Metropolitana
    'los bronces':                       'RM',
    # Region VI - O'Higgins
    'el teniente':                       'VI',
    # Region XV - Arica y Parinacota
    'pampa camarones':                   'XV',
}

REGION_NAMES = {
    'I':   'Tarapacá',
    'II':  'Antofagasta',
    'III': 'Atacama',
    'IV':  'Coquimbo',
    'V':   'Valparaíso',
    'RM':  'Metropolitana',
    'VI':  "O'Higgins",
    'XV':  'Arica y Parinacota',
}

REGION_ORDER = ['I', 'II', 'III', 'IV', 'V', 'RM', 'VI', 'XV']

print('Configuración cargada.')
print(f'  Orígenes de validación : {ORIGINS[0]}–{ORIGINS[-1]}  ({len(ORIGINS)} orígenes)')
print(f'  Horizontes             : H+{HORIZONS[0]}..H+{HORIZONS[-1]}')
print(f'  Regiones               : {REGION_ORDER}')
print(f'  Directorio outputs     : {OUT}')

Configuración cargada.
  Orígenes de validación : 2010–2018  (9 orígenes)
  Horizontes             : H+1..H+7
  Regiones               : ['I', 'II', 'III', 'IV', 'V', 'RM', 'VI', 'XV']
  Directorio outputs     : /Users/mac/TrabajoTesis/FinalResultsFolder/03_Forecasting/regional_model/outputs


## Sección 2 — Carga y Preparación de Datos

In [2]:
# ---------------------------------------------------------------------------
# 2.1  Cargar Produccion_Master.csv
# ---------------------------------------------------------------------------
prod_master = pd.read_csv(f"{BASE}/01_Data/processed/Produccion_Master.csv")
print(f"Produccion_Master: {prod_master.shape[0]} filas, {prod_master['Match_Key'].nunique()} minas")
print(f"  Años: {prod_master['Anio'].min()}–{prod_master['Anio'].max()}")

# Rename for clarity
prod_master = prod_master.rename(columns={'Match_Key': 'Mine', 'Anio': 'Year',
                                           'Produccion': 'Prod', 'Precio_Cobre': 'Cu_price'})

# ---------------------------------------------------------------------------
# 2.2  Cargar metadata
# ---------------------------------------------------------------------------
meta = pd.read_csv(f"{BASE}/01_Data/processed/metadata_minas.csv")
# Normalise column names
meta.columns = meta.columns.str.strip()
print(f"\nMetadata: {meta.shape[0]} filas")
print(meta[['Mine','Type','Owner']].to_string(index=False))

# ---------------------------------------------------------------------------
# 2.3  Asignar Región
# ---------------------------------------------------------------------------
prod_master['Region'] = prod_master['Mine'].map(MINE_REGION)
prod_master['Region_Name'] = prod_master['Region'].map(REGION_NAMES)

# Merge metadata
prod_master = prod_master.merge(
    meta[['Mine', 'Type', 'Owner', 'Start_Year', 'Holding']].drop_duplicates('Mine'),
    on='Mine', how='left'
)

# Mines without region mapping
no_region = prod_master[prod_master['Region'].isna()]['Mine'].unique()
if len(no_region) > 0:
    print(f"\nMinas sin región asignada: {sorted(no_region)}")

# ---------------------------------------------------------------------------
# 2.4  Filtrar: sólo minas con datos desde 1990 o antes (o que inicien >=1990)
#      y que tengan región asignada
# ---------------------------------------------------------------------------
# Keep only mines that appear in MINE_REGION
df = prod_master[prod_master['Region'].notna()].copy()

# Keep mines with at least one year >= 1990
mines_1990 = df[df['Year'] >= 1990].groupby('Mine')['Prod'].sum()
mines_1990 = mines_1990[mines_1990 > 0].index
df = df[df['Mine'].isin(mines_1990)].copy()

print(f"\nDespués de filtros: {df['Mine'].nunique()} minas, {df.shape[0]} filas")
print(f"Regiones presentes: {sorted(df['Region'].unique())}")

# ---------------------------------------------------------------------------
# 2.5  Tabla resumen: Region → Minas → Producción 2025
# ---------------------------------------------------------------------------
prod_2025 = df[df['Year'] == 2025].groupby(['Region', 'Mine'])['Prod'].sum().reset_index()
regional_2025 = prod_2025.groupby('Region').agg(
    N_Mines=('Mine', 'count'),
    Prod_2025_kt=('Prod', 'sum')
).reset_index()
regional_2025['Region_Name'] = regional_2025['Region'].map(REGION_NAMES)
regional_2025 = regional_2025.sort_values('Region')

print("\n--- Producción 2025 por Región (Miles TM) ---")
print(regional_2025[['Region', 'Region_Name', 'N_Mines', 'Prod_2025_kt']].to_string(index=False))
print(f"  TOTAL NACIONAL: {regional_2025['Prod_2025_kt'].sum():.1f} kt")

# Mines per region table
print("\n--- Minas por Región ---")
for reg in REGION_ORDER:
    mines_in_reg = sorted(df[df['Region'] == reg]['Mine'].unique())
    if mines_in_reg:
        print(f"  {reg} ({REGION_NAMES[reg]}): {mines_in_reg}")

Produccion_Master: 1246 filas, 36 minas
  Años: 1982–2025

Metadata: 42 filas
                   Mine      Type   Owner
              escondida  Sulfuros Private
                 spence     Mixto Private
         cerro colorado    Oxidos Private
             collahuasi  Sulfuros Private
            lomas bayas    Oxidos Private
            los bronces  Sulfuros Private
             el soldado  Sulfuros Private
                el abra    Oxidos Private
          los pelambres  Sulfuros Private
              centinela     Mixto Private
               antucoya    Oxidos Private
               zaldivar    Oxidos Private
            el teniente  Sulfuros Codelco
           chuquicamata  Sulfuros Codelco
         radomiro tomic    Oxidos Codelco
                 andina  Sulfuros Codelco
       gabriela mistral    Oxidos Codelco
         ministro hales  Sulfuros Codelco
               salvador  Sulfuros Codelco
        quebrada blanca     Mixto Private
              andacollo     Mixto Privat

## Sección 3 — Series de Tiempo Regionales

In [3]:
# ---------------------------------------------------------------------------
# 3.1  Precios nacionales de cobre (desde datos mensuales — precio real USD/lb)
# ---------------------------------------------------------------------------
monthly_data = pd.read_csv(f"{BASE}/01_Data/processed/1_master_thesis_data.csv")
monthly_data['Year'] = pd.to_datetime(monthly_data['Date']).dt.year
cu_prices = (monthly_data.groupby('Year')['Cu_Price'].mean()
             .rename('Cu_price'))

# Extend backwards using copper price index for pre-2014 years
# Use avg 2014-2018 as anchor to scale older years from Precio_Cobre if needed
# For now, backfill with earliest available (2014)
all_years = range(df['Year'].min(), df['Year'].max() + 1)
cu_prices = cu_prices.reindex(all_years).bfill().ffill()

print("Precios Cu reales (USc/lb) muestra:")
print(cu_prices.tail(12).to_string())


# ---------------------------------------------------------------------------
# 3.2  Codelco ownership flag
# ---------------------------------------------------------------------------
codelco_mines = set(df[df['Owner'] == 'Codelco']['Mine'].unique())
print(f"\nMinas Codelco: {sorted(codelco_mines)}")

# ---------------------------------------------------------------------------
# 3.3  Aggregate to regional annual totals
# ---------------------------------------------------------------------------
regional_ts = df.groupby(['Region', 'Year']).agg(
    Prod=('Prod', 'sum'),
    Total_Capital=('Capital_Stock', 'sum'),
    Total_Inv=('Inversion_MMUSD', 'sum'),
    N_Active=('Prod', lambda x: (x > 0).sum()),
    N_Codelco=('Mine', lambda x: sum(m in codelco_mines for m in x))
).reset_index()

# Merge national Cu_price
regional_ts = regional_ts.merge(cu_prices.rename('Cu_price').reset_index(),
                                 on='Year', how='left')

# Fill missing Cu_price via forward-fill
regional_ts = regional_ts.sort_values(['Region', 'Year'])
regional_ts['Cu_price'] = regional_ts.groupby('Region')['Cu_price'].ffill().bfill()

print(f"\nSeries regionales: {regional_ts.shape[0]} filas")
print(f"  Regiones: {sorted(regional_ts['Region'].unique())}")
print(f"  Años: {regional_ts['Year'].min()}–{regional_ts['Year'].max()}")

# ---------------------------------------------------------------------------
# 3.4  Tabla pivote: Region vs últimos 10 años
# ---------------------------------------------------------------------------
last10 = list(range(2016, 2026))
pivot_prod = regional_ts[regional_ts['Year'].isin(last10)].pivot(
    index='Region', columns='Year', values='Prod'
).reindex(index=REGION_ORDER)
pivot_prod.columns.name = None
pivot_prod.index.name = 'Region'
pivot_prod = pivot_prod.round(1).fillna(0.0)

print("\n--- Producción Regional (Miles TM) — 2016 a 2025 ---")
print(pivot_prod.to_string())

# National totals row
nat_row = regional_ts[regional_ts['Year'].isin(last10)].groupby('Year')['Prod'].sum()
print("\nNacional total:")
print(nat_row.round(1).to_string())

Precios Cu reales (USc/lb) muestra:
Year
2014    311.150167
2015    249.552333
2016    220.592167
2017    279.538500
2018    295.984333
2019    272.387583
2020    279.802667
2021    422.509500
2022    399.833417
2023    384.800750
2024    414.769083
2025    450.836667

Minas Codelco: ['andina', 'chuquicamata', 'el teniente', 'gabriela mistral', 'ministro hales', 'radomiro tomic', 'salvador']

Series regionales: 318 filas
  Regiones: ['I', 'II', 'III', 'IV', 'RM', 'V', 'VI', 'XV']
  Años: 1982–2025

--- Producción Regional (Miles TM) — 2016 a 2025 ---
          2016    2017    2018    2019    2020    2021    2022    2023    2024    2025
Region                                                                                
I        615.2   613.6   650.9   658.2   711.4   699.0   631.6   672.9   766.4   596.0
II      2998.6  2934.9  3201.2  3189.3  3145.1  3089.9  3058.6  3028.6  3212.0  3356.5
III      380.6   390.9   357.6   368.2   332.1   347.0   347.9   334.4   345.7   415.9
IV      

## Sección 4 — Feature Engineering

In [4]:
# ---------------------------------------------------------------------------
# 4.1  Region_Size: quartile of average production 2000-2018 (fixed, no leakage)
# ---------------------------------------------------------------------------
region_avg_prod = (
    regional_ts[(regional_ts['Year'] >= 2000) & (regional_ts['Year'] <= 2009)]
    .groupby('Region')['Prod'].mean()
)
q25, q50, q75 = region_avg_prod.quantile([0.25, 0.50, 0.75])
print(f"Region_Size quartiles (avg prod 2000–2009): q25={q25:.1f}, q50={q50:.1f}, q75={q75:.1f}")

def assign_region_size(avg_prod):
    if avg_prod <= q25:
        return 0  # Small
    elif avg_prod <= q50:
        return 1  # Medium
    elif avg_prod <= q75:
        return 2  # Large
    else:
        return 3  # Colossal

region_size_map = region_avg_prod.apply(assign_region_size).to_dict()
size_labels = {0: 'Small', 1: 'Medium', 2: 'Large', 3: 'Colossal'}
print("\nRegion_Size assignment:")
for reg in REGION_ORDER:
    rs = region_size_map.get(reg, np.nan)
    avg = region_avg_prod.get(reg, 0.0)
    print(f"  {reg:4s} ({REGION_NAMES[reg]:20s}): avg={avg:7.1f} kt → Size={rs} ({size_labels.get(rs,'?')})")

# ---------------------------------------------------------------------------
# 4.2  Cu_regime: 10-year rolling percentile of Cu price at origin (no leakage)
# ---------------------------------------------------------------------------
all_years = sorted(cu_prices.index)

def cu_regime_at(year, window=10):
    """Percentile of Cu_price at `year` within [year-window+1, year]."""
    yrs = [y for y in all_years if y >= year - window + 1 and y <= year]
    prices = cu_prices[yrs]
    if len(prices) < 2:
        return 0.5
    p = (prices < cu_prices[year]).sum() / (len(prices) - 1)
    return float(np.clip(p, 0.0, 1.0))

cu_regime_cache = {y: cu_regime_at(y) for y in range(1995, 2026)}
print("\nCu_regime (last 5 years):")
for y in range(2021, 2026):
    print(f"  {y}: {cu_regime_cache[y]:.3f}")

# ---------------------------------------------------------------------------
# 4.3  Helper: build feature vector for (region, origin_year, h)
# ---------------------------------------------------------------------------
def build_features_for_row(region, origin_year, h, ts_up_to_origin):
    """
    Compute feature vector for a single (region, origin_year, h) observation.
    ts_up_to_origin: regional_ts filtered to year <= origin_year.
    """
    r_ts = ts_up_to_origin[ts_up_to_origin['Region'] == region].sort_values('Year')

    if r_ts.empty or origin_year not in r_ts['Year'].values:
        return None

    row_origin = r_ts[r_ts['Year'] == origin_year].iloc[0]
    prod_origin = row_origin['Prod']

    if prod_origin <= 0:
        return None

    # Prod_origin_minus1
    prev = r_ts[r_ts['Year'] == origin_year - 1]
    prod_prev = prev['Prod'].values[0] if not prev.empty and prev['Prod'].values[0] > 0 else prod_origin

    # Prod_Lag1 = log(prod_origin / prod_prev)
    prod_lag1 = np.log(prod_origin / prod_prev) if prod_prev > 0 else 0.0

    # Prod_pct_change: clipped [-2, 2]
    pct_chg = (prod_origin - prod_prev) / prod_prev if prod_prev > 0 else 0.0
    pct_chg = float(np.clip(pct_chg, -2.0, 2.0))

    # Tendencia_5y: slope of linear regression on log(Prod) over [origin-4..origin]
    trend_years = r_ts[(r_ts['Year'] >= origin_year - 4) & (r_ts['Year'] <= origin_year)]
    if len(trend_years) >= 3 and (trend_years['Prod'] > 0).all():
        log_prods = np.log(trend_years['Prod'].values)
        slope, _, _, _, _ = linregress(range(len(log_prods)), log_prods)
        tendencia_5y = float(slope)
    else:
        tendencia_5y = 0.0

    # Region_Size
    region_size = region_size_map.get(region, 1)

    # N_Active and Capital_per_Mine
    n_active = int(row_origin['N_Active'])
    total_cap = row_origin['Total_Capital']
    cap_per_mine = total_cap / n_active if n_active > 0 else 0.0

    # Owner_share: fraction Codelco mines
    n_codelco = int(row_origin['N_Codelco'])
    owner_share = n_codelco / n_active if n_active > 0 else 0.0

    # Cu_regime
    cu_reg = cu_regime_cache.get(origin_year, 0.5)

    # Is_Pandemic_Target
    target_year = origin_year + h
    is_pandemic = 1 if target_year in {2020, 2021} else 0

    return {
        'Region':           region,
        'Origin_Year':      origin_year,
        'Horizonte_feat':   h,
        'Target_Year':      target_year,
        'Origin_Prod':      prod_origin,
        'Region_Size':      region_size,
        'Prod_Lag1':        prod_lag1,
        'Tendencia_5y':     tendencia_5y,
        'Prod_pct_change':  pct_chg,
        'Cu_regime':        cu_reg,
        'N_Active':         n_active,
        'Capital_per_Mine': cap_per_mine,
        'Is_Pandemic_Target': is_pandemic,
        'Owner_share':      owner_share,
    }

FEATURE_COLS = [
    'Region_Size', 'Prod_Lag1', 'Tendencia_5y', 'Prod_pct_change',
    'Cu_regime', 'N_Active', 'Capital_per_Mine', 'Is_Pandemic_Target',
    'Horizonte_feat', 'Owner_share'
]

print("\nFeatures usadas en modelos:")
for i, f in enumerate(FEATURE_COLS, 1):
    print(f"  {i:2d}. {f}")

Region_Size quartiles (avg prod 2000–2009): q25=268.1, q50=340.0, q75=515.5

Region_Size assignment:
  I    (Tarapacá            ): avg=  644.5 kt → Size=3 (Colossal)
  II   (Antofagasta         ): avg= 2746.8 kt → Size=3 (Colossal)
  III  (Atacama             ): avg=  257.7 kt → Size=0 (Small)
  IV   (Coquimbo            ): avg=  340.0 kt → Size=1 (Medium)
  V    (Valparaíso          ): avg=  233.8 kt → Size=0 (Small)
  RM   (Metropolitana       ): avg=  278.5 kt → Size=1 (Medium)
  VI   (O'Higgins           ): avg=  386.6 kt → Size=2 (Large)
  XV   (Arica y Parinacota  ): avg=    0.0 kt → Size=nan (?)

Cu_regime (last 5 years):
  2021: 1.000
  2022: 0.889
  2023: 0.778
  2024: 0.889
  2025: 1.000

Features usadas en modelos:
   1. Region_Size
   2. Prod_Lag1
   3. Tendencia_5y
   4. Prod_pct_change
   5. Cu_regime
   6. N_Active
   7. Capital_per_Mine
   8. Is_Pandemic_Target
   9. Horizonte_feat
  10. Owner_share


In [5]:
# ---------------------------------------------------------------------------
# 4.4  Build full validation dataset
# ---------------------------------------------------------------------------
print("Construyendo dataset de validación...")
rows = []

for origin in ORIGINS:
    ts_train = regional_ts[regional_ts['Year'] <= origin].copy()
    for reg in sorted(regional_ts['Region'].unique()):
        for h in HORIZONS:
            target_year = origin + h
            # Check target exists
            target_row = regional_ts[
                (regional_ts['Region'] == reg) & (regional_ts['Year'] == target_year)
            ]
            if target_row.empty:
                continue
            prod_target = target_row['Prod'].values[0]

            feat_dict = build_features_for_row(reg, origin, h, ts_train)
            if feat_dict is None:
                continue

            prod_origin = feat_dict['Origin_Prod']
            if prod_origin <= 0 or prod_target < 0:
                continue

            log_ratio = np.log(prod_target / prod_origin) if prod_target > 0 else np.log(0.01)
            log_ratio = float(np.clip(log_ratio, -3.0, 3.0))

            feat_dict['LogRatio'] = log_ratio
            feat_dict['Prod_Target'] = prod_target
            rows.append(feat_dict)

val_df = pd.DataFrame(rows)
print(f"Dataset validación: {val_df.shape[0]} filas")
print(f"  Regiones: {sorted(val_df['Region'].unique())}")
print(f"  Orígenes: {sorted(val_df['Origin_Year'].unique())}")
print(f"  Horizontes: {sorted(val_df['Horizonte_feat'].unique())}")
print(f"\nEstadísticas LogRatio:")
print(val_df['LogRatio'].describe().round(4).to_string())

Construyendo dataset de validación...


Dataset validación: 456 filas
  Regiones: ['I', 'II', 'III', 'IV', 'RM', 'V', 'VI', 'XV']
  Orígenes: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]
  Horizontes: [1, 2, 3, 4, 5, 6, 7]

Estadísticas LogRatio:
count    456.0000
mean       0.0379
std        0.2505
min       -1.0450
25%       -0.0901
50%        0.0097
75%        0.1058
max        1.3113


## Sección 5 — Validación Rolling-Origin

In [6]:
# ---------------------------------------------------------------------------
# 5.1  Optuna tuning helpers
# ---------------------------------------------------------------------------

def tune_xgb(X_train, y_train, n_trials=N_TRIALS, random_state=RANDOM_STATE):
    """Tune XGBoost hyperparameters with Optuna (5-fold CV on train)."""
    from sklearn.model_selection import cross_val_score

    def objective(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 100, 1000),
            'max_depth':         trial.suggest_int('max_depth', 2, 8),
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight':  trial.suggest_int('min_child_weight', 1, 20),
            'random_state':      random_state,
            'verbosity':         0,
            'n_jobs':            -1,
        }
        model = xgb.XGBRegressor(**params)
        scores = cross_val_score(model, X_train, y_train,
                                 cv=5, scoring='neg_mean_absolute_error')
        return -scores.mean()

    study = optuna.create_study(direction='minimize',
                                sampler=TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params


def tune_lgb(X_train, y_train, n_trials=N_TRIALS, random_state=RANDOM_STATE):
    """Tune LightGBM hyperparameters with Optuna (5-fold CV on train)."""
    from sklearn.model_selection import cross_val_score

    def objective(trial):
        params = {
            'n_estimators':       trial.suggest_int('n_estimators', 100, 1000),
            'max_depth':          trial.suggest_int('max_depth', 2, 8),
            'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':          trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'num_leaves':         trial.suggest_int('num_leaves', 10, 100),
            'min_child_samples':  trial.suggest_int('min_child_samples', 5, 50),
            'random_state':       random_state,
            'verbosity':          -1,
            'n_jobs':             -1,
        }
        model = lgb.LGBMRegressor(**params)
        scores = cross_val_score(model, X_train, y_train,
                                 cv=5, scoring='neg_mean_absolute_error')
        return -scores.mean()

    study = optuna.create_study(direction='minimize',
                                sampler=TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params


print("Funciones de tuning definidas.")

Funciones de tuning definidas.


In [7]:
# ---------------------------------------------------------------------------
# 5.2  Tune XGBoost using ALL training data combined (pooled across origins)
# ---------------------------------------------------------------------------
# For tuning we use all observations from origins < max(ORIGINS)
# to find good global hyperparameters
tune_mask = val_df['Origin_Year'] <= ORIGINS[-2]   # use origins 2010–2017 for tuning
X_tune = val_df.loc[tune_mask, FEATURE_COLS].values
y_tune = val_df.loc[tune_mask, 'LogRatio'].values

print(f"Tuning set: {len(y_tune)} observaciones (orígenes 2010–2017)")

print("\n[XGBoost] Iniciando Optuna ({N_TRIALS} trials)...".format(N_TRIALS=N_TRIALS))
best_xgb_params = tune_xgb(X_tune, y_tune, n_trials=N_TRIALS)
print(f"  Mejores params XGB: {best_xgb_params}")

print("\n[LightGBM] Iniciando Optuna ({N_TRIALS} trials)...".format(N_TRIALS=N_TRIALS))
best_lgb_params = tune_lgb(X_tune, y_tune, n_trials=N_TRIALS)
print(f"  Mejores params LGB: {best_lgb_params}")

# Add fixed kwargs
best_xgb_params['random_state'] = RANDOM_STATE
best_xgb_params['verbosity']    = 0
best_xgb_params['n_jobs']       = -1

best_lgb_params['random_state'] = RANDOM_STATE
best_lgb_params['verbosity']    = -1
best_lgb_params['n_jobs']       = -1

print("\nTuning completado.")

Tuning set: 407 observaciones (orígenes 2010–2017)

[XGBoost] Iniciando Optuna (50 trials)...


  Mejores params XGB: {'n_estimators': 347, 'max_depth': 5, 'learning_rate': 0.04031285876535082, 'subsample': 0.5321528533673582, 'colsample_bytree': 0.543258470309539, 'min_child_weight': 3}

[LightGBM] Iniciando Optuna (50 trials)...


  Mejores params LGB: {'n_estimators': 513, 'max_depth': 2, 'learning_rate': 0.04634598796674968, 'subsample': 0.7915585123873445, 'colsample_bytree': 0.8478684057508008, 'num_leaves': 17, 'min_child_samples': 6}

Tuning completado.


In [8]:
# ---------------------------------------------------------------------------
# 5.3  Rolling-origin validation loop
# ---------------------------------------------------------------------------
print("=" * 60)
print("ROLLING-ORIGIN VALIDATION")
print("=" * 60)

val_results = []   # one row per (region, origin, h)

for origin in ORIGINS:
    # Train: all observations where Origin_Year < current origin (expanding)
    # (strict: train on years BEFORE this origin)
    train_mask = val_df['Origin_Year'] < origin
    test_mask  = val_df['Origin_Year'] == origin

    if train_mask.sum() < 5:
        print(f"  Origin {origin}: insuficientes datos de train ({train_mask.sum()} filas), salteando.")
        continue

    X_train = val_df.loc[train_mask, FEATURE_COLS].values
    y_train = val_df.loc[train_mask, 'LogRatio'].values
    X_test  = val_df.loc[test_mask,  FEATURE_COLS].values

    if len(X_test) == 0:
        continue

    # Fit models
    xgb_model = xgb.XGBRegressor(**best_xgb_params)
    xgb_model.fit(X_train, y_train)

    lgb_model = lgb.LGBMRegressor(**best_lgb_params)
    lgb_model.fit(X_train, y_train)

    # Predictions
    xgb_preds = xgb_model.predict(X_test)
    lgb_preds = lgb_model.predict(X_test)
    ens_preds = 0.5 * xgb_preds + 0.5 * lgb_preds

    # Store results
    test_rows = val_df[test_mask].copy().reset_index(drop=True)
    test_rows['XGB_pred']  = xgb_preds
    test_rows['LGB_pred']  = lgb_preds
    test_rows['Ens_pred']  = ens_preds
    test_rows['Naive_pred'] = 0.0   # LogRatio=0 → flat

    # Errors in LogRatio space
    for m in ['XGB', 'LGB', 'Ens']:
        test_rows[f'{m}_err'] = np.abs(test_rows[f'{m}_pred'] - test_rows['LogRatio'])
    test_rows['Naive_err'] = np.abs(test_rows['Naive_pred'] - test_rows['LogRatio'])

    val_results.append(test_rows)
    n_reg = test_rows['Region'].nunique()
    print(f"  Origin {origin}: {len(test_rows):3d} obs ({n_reg} regiones), "
          f"XGB_MAE={test_rows['XGB_err'].mean():.4f}, "
          f"LGB_MAE={test_rows['LGB_err'].mean():.4f}, "
          f"Naive_MAE={test_rows['Naive_err'].mean():.4f}")

val_all = pd.concat(val_results, ignore_index=True)
print(f"\nTotal predicciones validación: {val_all.shape[0]}")

ROLLING-ORIGIN VALIDATION
  Origin 2010: insuficientes datos de train (0 filas), salteando.


  Origin 2011:  49 obs (7 regiones), XGB_MAE=0.1273, LGB_MAE=0.1163, Naive_MAE=0.1831


  Origin 2012:  49 obs (7 regiones), XGB_MAE=0.1488, LGB_MAE=0.1243, Naive_MAE=0.1913


  Origin 2013:  49 obs (7 regiones), XGB_MAE=0.1193, LGB_MAE=0.1346, Naive_MAE=0.1269


  Origin 2014:  54 obs (8 regiones), XGB_MAE=0.1665, LGB_MAE=0.1342, Naive_MAE=0.1549


  Origin 2015:  54 obs (8 regiones), XGB_MAE=0.2044, LGB_MAE=0.2137, Naive_MAE=0.1345


  Origin 2016:  54 obs (8 regiones), XGB_MAE=0.2042, LGB_MAE=0.2155, Naive_MAE=0.1683


  Origin 2017:  49 obs (7 regiones), XGB_MAE=0.1511, LGB_MAE=0.1466, Naive_MAE=0.1227


  Origin 2018:  49 obs (7 regiones), XGB_MAE=0.1674, LGB_MAE=0.1253, Naive_MAE=0.1375

Total predicciones validación: 407


## Sección 6 — Métricas y Scoreboard

In [9]:
# ---------------------------------------------------------------------------
# 6.1  Helper: compute metrics for a model column
# ---------------------------------------------------------------------------

def compute_metrics(df_in, pred_col, naive_col='Naive_pred', target_col='LogRatio'):
    """
    Returns dict with WR, MAE_Model, MAE_Naive, Skill%, MASE.
    Win Rate: % of pairs where |model_error| < |naive_error|.
    """
    err_model = np.abs(df_in[pred_col] - df_in[target_col])
    err_naive = np.abs(df_in[naive_col] - df_in[target_col])

    n = len(err_model)
    if n == 0:
        return {'WR': np.nan, 'MAE_Model': np.nan, 'MAE_Naive': np.nan,
                'Skill': np.nan, 'MASE': np.nan, 'N': 0}

    wr     = (err_model < err_naive).sum() / n
    mae_m  = err_model.mean()
    mae_n  = err_naive.mean()
    skill  = (1 - mae_m / mae_n) * 100 if mae_n > 0 else 0.0
    mase   = mae_m / mae_n if mae_n > 0 else np.nan

    return {'WR': round(wr, 4), 'MAE_Model': round(mae_m, 4),
            'MAE_Naive': round(mae_n, 4), 'Skill': round(skill, 2),
            'MASE': round(mase, 4), 'N': n}


# ---------------------------------------------------------------------------
# 6.2  Per-region scoreboard
# ---------------------------------------------------------------------------
models = ['XGB', 'LGB', 'Ens']
model_col_map = {'XGB': 'XGB_pred', 'LGB': 'LGB_pred', 'Ens': 'Ens_pred'}

scoreboard_rows = []

for reg in sorted(val_all['Region'].unique()):
    reg_df = val_all[val_all['Region'] == reg]
    for model_name in models:
        m = compute_metrics(reg_df, model_col_map[model_name])
        m['Region']      = reg
        m['Region_Name'] = REGION_NAMES.get(reg, reg)
        m['Model']       = model_name
        scoreboard_rows.append(m)

# National aggregate
for model_name in models:
    m = compute_metrics(val_all, model_col_map[model_name])
    m['Region']      = 'NAC'
    m['Region_Name'] = 'Nacional'
    m['Model']       = model_name
    scoreboard_rows.append(m)

scoreboard = pd.DataFrame(scoreboard_rows)
scoreboard['WR_pct'] = (scoreboard['WR'] * 100).round(1)

print("\n=" * 60)
print("SCOREBOARD POR REGIÓN Y MODELO (ordenado por Skill%)")
print("=" * 60)
display_cols = ['Region', 'Region_Name', 'Model', 'WR_pct', 'MAE_Model',
                'MAE_Naive', 'Skill', 'MASE', 'N']
print(scoreboard.sort_values(['Region', 'Skill'], ascending=[True, False])
      [display_cols].to_string(index=False))

# ---------------------------------------------------------------------------
# 6.3  Per-horizon WR table
# ---------------------------------------------------------------------------
print("\n--- Win Rate por Horizonte (todos los modelos, todas las regiones) ---")
hor_rows = []
for h in HORIZONS:
    h_df = val_all[val_all['Horizonte_feat'] == h]
    row = {'Horizonte': h}
    for model_name in models:
        m = compute_metrics(h_df, model_col_map[model_name])
        row[f'WR_{model_name}'] = round(m['WR'] * 100, 1)
    hor_rows.append(row)
hor_table = pd.DataFrame(hor_rows)
print(hor_table.to_string(index=False))


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
SCOREBOARD POR REGIÓN Y MODELO (ordenado por Skill%)
Region        Region_Name Model  WR_pct  MAE_Model  MAE_Naive  Skill   MASE   N
     I           Tarapacá   XGB    51.8     0.1174     0.1297   9.52 0.9048  56
     I           Tarapacá   Ens    48.2     0.1205     0.1297   7.07 0.9293  56
     I           Tarapacá   LGB    50.0     0.1336     0.1297  -2.97 1.0297  56
    II        Antofagasta   XGB    44.6     0.0668     0.0486 -37.50 1.3750  56
    II        Antofagasta   Ens    41.1     0.0672     0.0486 -38.36 1.3836  56
    II        Antofagasta   LGB    35.7     0.0735     0.0486 -51.26 1.5126  56
   III            Atacama   Ens    73.2     0.1388     0.2315  40.06 0.5994  56
   III            Atacama   LGB    71.4     0.1418     0.2315  38.76 0.6124  56
   III            Atacama   XGB    71.4     0.1470     0.2315  36.51 0.6349  56
    IV           Coquimbo 

## Sección 7 — Ensemble y Selección del Mejor Modelo

In [10]:
# ---------------------------------------------------------------------------
# 7.1  Ensemble already computed (Ens_pred = 0.5*XGB + 0.5*LGB).
#      Now select best model by national Skill%.
# ---------------------------------------------------------------------------
nat_scores = scoreboard[scoreboard['Region'] == 'NAC'][['Model', 'WR_pct', 'Skill', 'MASE']]
nat_scores = nat_scores.sort_values('Skill', ascending=False)

print("Métricas NACIONALES:")
print(nat_scores.to_string(index=False))

best_model_name = nat_scores.iloc[0]['Model']
best_pred_col   = model_col_map[best_model_name]
print(f"\nMejor modelo nacional: {best_model_name} "
      f"(Skill={nat_scores.iloc[0]['Skill']}%, "
      f"WR={nat_scores.iloc[0]['WR_pct']}%)")

# ---------------------------------------------------------------------------
# 7.2  Per-region best model
# ---------------------------------------------------------------------------
print("\n--- Mejor modelo por Región ---")
best_per_region = (
    scoreboard[scoreboard['Region'] != 'NAC']
    .sort_values('Skill', ascending=False)
    .groupby('Region')
    .first()
    .reset_index()
)
print(best_per_region[['Region', 'Region_Name', 'Model', 'WR_pct', 'Skill', 'MASE']]
      .to_string(index=False))

Métricas NACIONALES:
Model  WR_pct  Skill   MASE
  LGB    51.8  -0.16 1.0016
  Ens    47.9  -0.59 1.0059
  XGB    43.5  -6.45 1.0645

Mejor modelo nacional: LGB (Skill=-0.16%, WR=51.8%)

--- Mejor modelo por Región ---
Region        Region_Name Model  WR_pct  Skill   MASE
     I           Tarapacá   XGB    51.8   9.52 0.9048
    II        Antofagasta   XGB    44.6 -37.50 1.3750
   III            Atacama   Ens    73.2  40.06 0.5994
    IV           Coquimbo   XGB    33.9 -54.73 1.5473
    RM      Metropolitana   XGB    48.2  -8.83 1.0883
     V         Valparaíso   LGB    53.6  -5.04 1.0504
    VI          O'Higgins   LGB    67.9  20.17 0.7983
    XV Arica y Parinacota   LGB    46.7  20.99 0.7901


## Sección 8 — Proyecciones 2026-2032

In [11]:
# ---------------------------------------------------------------------------
# 8.1  Train final models using ALL available data (origin <= 2025)
# ---------------------------------------------------------------------------
print("Entrenando modelos finales con todos los datos disponibles...")

# All validation rows as training data for final model
X_all = val_df[FEATURE_COLS].values
y_all = val_df['LogRatio'].values

xgb_final = xgb.XGBRegressor(**best_xgb_params)
xgb_final.fit(X_all, y_all)

lgb_final = lgb.LGBMRegressor(**best_lgb_params)
lgb_final.fit(X_all, y_all)

print("Modelos finales entrenados.")

# ---------------------------------------------------------------------------
# 8.2  Quantile models for uncertainty (q10/q90)
# ---------------------------------------------------------------------------
lgb_q10_params = {**best_lgb_params, 'objective': 'quantile', 'alpha': 0.10}
lgb_q90_params = {**best_lgb_params, 'objective': 'quantile', 'alpha': 0.90}
lgb_q10_params.pop('n_jobs', None)
lgb_q90_params.pop('n_jobs', None)
lgb_q10_params.pop('verbosity', None)
lgb_q90_params.pop('verbosity', None)

lgb_q10 = lgb.LGBMRegressor(**lgb_q10_params, verbosity=-1)
lgb_q90 = lgb.LGBMRegressor(**lgb_q90_params, verbosity=-1)
lgb_q10.fit(X_all, y_all)
lgb_q90.fit(X_all, y_all)

print("Modelos cuantílicos (q10/q90) entrenados.")

# ---------------------------------------------------------------------------
# 8.3  Build projection features (origin=2025, h=1..7)
# ---------------------------------------------------------------------------
print("\nConstruyendo features de proyección (origen=2025)...")

ts_2025 = regional_ts[regional_ts['Year'] <= 2025].copy()

proj_rows = []
for reg in sorted(regional_ts['Region'].unique()):
    for h in HORIZONS:
        feat_dict = build_features_for_row(reg, 2025, h, ts_2025)
        if feat_dict is None:
            continue
        # No pandemic in 2026-2032
        feat_dict['Is_Pandemic_Target'] = 0
        proj_rows.append(feat_dict)

proj_df = pd.DataFrame(proj_rows)
print(f"Proyecciones: {len(proj_df)} filas ({proj_df['Region'].nunique()} regiones × {len(HORIZONS)} horizontes)")

# ---------------------------------------------------------------------------
# 8.4  Predict projections
# ---------------------------------------------------------------------------
X_proj = proj_df[FEATURE_COLS].values

proj_df['LogRatio_XGB']  = xgb_final.predict(X_proj)
proj_df['LogRatio_LGB']  = lgb_final.predict(X_proj)
proj_df['LogRatio_Ens']  = 0.5 * proj_df['LogRatio_XGB'] + 0.5 * proj_df['LogRatio_LGB']
proj_df['LogRatio_q10']  = lgb_q10.predict(X_proj)
proj_df['LogRatio_q90']  = lgb_q90.predict(X_proj)

# Recover kt from LogRatio
best_lr_col = {'XGB': 'LogRatio_XGB', 'LGB': 'LogRatio_LGB', 'Ens': 'LogRatio_Ens'}[best_model_name]

proj_df['Pred']       = np.exp(proj_df[best_lr_col]) * proj_df['Origin_Prod']
proj_df['Naive_Pred'] = proj_df['Origin_Prod']   # flat
proj_df['Lower']      = np.exp(proj_df['LogRatio_q10']) * proj_df['Origin_Prod']
proj_df['Upper']      = np.exp(proj_df['LogRatio_q90']) * proj_df['Origin_Prod']

# Ensure Lower <= Pred <= Upper
proj_df['Lower'] = proj_df[['Lower', 'Pred']].min(axis=1)
proj_df['Upper'] = proj_df[['Upper', 'Pred']].max(axis=1)

proj_df['ForecastYear'] = proj_df['Origin_Year'] + proj_df['Horizonte_feat']
proj_df['Region_Name']  = proj_df['Region'].map(REGION_NAMES)

# ---------------------------------------------------------------------------
# 8.5  Display projections table
# ---------------------------------------------------------------------------
print("\n--- Proyecciones Regionales 2026-2032 (kt, mejor modelo: {}) ---".format(best_model_name))
pivot_proj = proj_df.pivot_table(
    index='Region', columns='ForecastYear', values='Pred', aggfunc='first'
).reindex(index=sorted(proj_df['Region'].unique()))
pivot_proj.columns.name = None
print(pivot_proj.round(1).to_string())

print("\n--- Naive (producción 2025, flat) ---")
naive_table = proj_df[proj_df['Horizonte_feat'] == 1][['Region', 'Region_Name', 'Naive_Pred']]
print(naive_table.to_string(index=False))

Entrenando modelos finales con todos los datos disponibles...


Modelos finales entrenados.


Modelos cuantílicos (q10/q90) entrenados.

Construyendo features de proyección (origen=2025)...
Proyecciones: 56 filas (8 regiones × 7 horizontes)

--- Proyecciones Regionales 2026-2032 (kt, mejor modelo: LGB) ---
          2026    2027    2028    2029    2030    2031    2032
Region                                                        
I        807.2   817.6   810.0   787.3   759.6   745.7   737.7
II      3296.1  3296.1  3296.1  3226.3  3139.5  3123.9  3102.1
III      418.7   402.2   412.9   405.9   399.7   398.9   396.5
IV       309.9   309.9   309.9   304.1   295.2   292.8   291.0
RM       206.0   208.6   208.6   205.9   199.1   197.5   193.8
V        213.3   215.0   220.7   218.4   214.3   212.5   208.6
VI       315.7   319.8   319.8   315.6   305.2   302.7   297.1
XV         4.5     4.4     4.4     4.4     4.7     4.8     4.8

--- Naive (producción 2025, flat) ---
Region        Region_Name  Naive_Pred
     I           Tarapacá   596.04300
    II        Antofagasta  3356.49392
   

In [12]:
# ---------------------------------------------------------------------------
# 8.6  National aggregate projections
# ---------------------------------------------------------------------------
nat_proj = proj_df.groupby('ForecastYear').agg(
    Pred=('Pred', 'sum'),
    Naive_Pred=('Naive_Pred', 'sum'),
    Lower=('Lower', 'sum'),
    Upper=('Upper', 'sum'),
    Origin_Prod=('Origin_Prod', 'sum')
).reset_index()
nat_proj['Region']      = 'NAC'
nat_proj['Region_Name'] = 'Nacional'
nat_proj['Horizonte']   = nat_proj['ForecastYear'] - 2025
nat_proj['Region_Size'] = 3

print("\n--- Proyecciones NACIONALES 2026-2032 (kt) ---")
print("Modelo: {}  |  Origen: 2025".format(best_model_name))
print(nat_proj[['ForecastYear', 'Pred', 'Naive_Pred', 'Lower', 'Upper']].round(1).to_string(index=False))

prod_2025_total = nat_proj['Origin_Prod'].iloc[0]
print(f"\nProducción 2025 (total nacional): {prod_2025_total:.1f} kt")

print("\n--- Cambio % respecto a 2025 ---")
nat_proj_pct = nat_proj.copy()
nat_proj_pct['Pred_pct']  = ((nat_proj_pct['Pred']  / prod_2025_total - 1) * 100).round(1)
nat_proj_pct['Naive_pct'] = 0.0
nat_proj_pct['Lower_pct'] = ((nat_proj_pct['Lower'] / prod_2025_total - 1) * 100).round(1)
nat_proj_pct['Upper_pct'] = ((nat_proj_pct['Upper'] / prod_2025_total - 1) * 100).round(1)
print(nat_proj_pct[['ForecastYear', 'Pred_pct', 'Lower_pct', 'Upper_pct']].to_string(index=False))


--- Proyecciones NACIONALES 2026-2032 (kt) ---
Modelo: LGB  |  Origen: 2025
 ForecastYear   Pred  Naive_Pred  Lower  Upper
         2026 5571.5      5471.8 5338.2 6073.8
         2027 5573.6      5471.8 5289.6 6088.7
         2028 5582.3      5471.8 5307.6 6107.2
         2029 5468.0      5471.8 5197.4 6113.5
         2030 5317.4      5471.8 5056.4 6105.1
         2031 5278.8      5471.8 4888.9 6164.9
         2032 5231.5      5471.8 4798.1 6163.9

Producción 2025 (total nacional): 5471.8 kt

--- Cambio % respecto a 2025 ---
 ForecastYear  Pred_pct  Lower_pct  Upper_pct
         2026       1.8       -2.4       11.0
         2027       1.9       -3.3       11.3
         2028       2.0       -3.0       11.6
         2029      -0.1       -5.0       11.7
         2030      -2.8       -7.6       11.6
         2031      -3.5      -10.7       12.7
         2032      -4.4      -12.3       12.6


## Sección 9 — Comparación Bottom-Up

In [13]:
# ---------------------------------------------------------------------------
# 9.1  Load mine-level ML projections and scoreboard
# ---------------------------------------------------------------------------
mine_proj = pd.read_csv(f"{BASE}/03_Forecasting/03_annual_model/outputs_best/projections_2026_2032.csv")
mine_score = pd.read_csv(f"{BASE}/03_Forecasting/03_annual_model/outputs_best/scoreboard_annual_v7.csv")

print(f"Mine projections: {mine_proj.shape[0]} filas, {mine_proj['Mine'].nunique()} minas")
print(f"Mine scoreboard : {mine_score.shape[0]} minas")

# ---------------------------------------------------------------------------
# 9.2  Decide per-mine: use ML pred or naive
#      Rule: use Naive_Pred if verdict=='NAIVE★' OR WR < 0.50
# ---------------------------------------------------------------------------
mine_score['use_naive'] = (
    (mine_score['verdict'] == 'NAIVE★') | (mine_score['WR'] < 0.50)
)
naive_mines = set(mine_score[mine_score['use_naive']]['Mine'].unique())
print(f"\nMinas con naive preferido: {sorted(naive_mines)}")

# Merge decision into projections
mine_proj_adj = mine_proj.merge(
    mine_score[['Mine', 'use_naive', 'verdict']],
    on='Mine', how='left'
)
mine_proj_adj['use_naive'] = mine_proj_adj['use_naive'].fillna(False)

# Adjusted prediction
mine_proj_adj['Pred_adj'] = np.where(
    mine_proj_adj['use_naive'],
    mine_proj_adj['Naive_Pred'],
    mine_proj_adj['Pred']
)

# ---------------------------------------------------------------------------
# 9.3  Add excluded mines (spence, quebrada blanca) at flat naive
# ---------------------------------------------------------------------------
excluded_mines = ['spence', 'quebrada blanca']
excluded_rows = []

for exc_mine in excluded_mines:
    exc_prod_2025 = df[(df['Mine'] == exc_mine) & (df['Year'] == 2025)]['Prod'].sum()
    exc_region = MINE_REGION.get(exc_mine, 'II')
    if exc_prod_2025 > 0:
        for h in HORIZONS:
            excluded_rows.append({
                'Mine':         exc_mine,
                'ForecastYear': 2025 + h,
                'Horizonte':    h,
                'Pred':         exc_prod_2025,
                'Naive_Pred':   exc_prod_2025,
                'Pred_adj':     exc_prod_2025,
                'Origin_Prod':  exc_prod_2025,
                'use_naive':    True,
                'verdict':      'NAIVE★ (excl)',
            })
        print(f"  {exc_mine}: Prod_2025={exc_prod_2025:.1f} kt → naive flat, Region={exc_region}")
    else:
        print(f"  {exc_mine}: sin producción 2025, ignorando.")

if excluded_rows:
    excl_df = pd.DataFrame(excluded_rows)
    # Assign region
    excl_df['Region'] = excl_df['Mine'].map(MINE_REGION)
    mine_proj_adj['Region'] = mine_proj_adj['Mine'].map(MINE_REGION)
    mine_proj_adj = pd.concat([mine_proj_adj, excl_df], ignore_index=True)
else:
    mine_proj_adj['Region'] = mine_proj_adj['Mine'].map(MINE_REGION)

print(f"\nMines proyectadas (incl. excluidas): {mine_proj_adj['Mine'].nunique()}")

Mine projections: 217 filas, 31 minas
Mine scoreboard : 24 minas

Minas con naive preferido: ['chuquicamata', 'collahuasi', 'el abra', 'los bronces', 'los pelambres', 'radomiro tomic']
  spence: Prod_2025=254.8 kt → naive flat, Region=II
  quebrada blanca: Prod_2025=190.0 kt → naive flat, Region=I

Mines proyectadas (incl. excluidas): 33


In [14]:
# ---------------------------------------------------------------------------
# 9.4  Bottom-up: aggregate mine → region → national
# ---------------------------------------------------------------------------
bu_regional = mine_proj_adj.groupby(['Region', 'ForecastYear']).agg(
    BU_Pred=('Pred_adj', 'sum'),
    BU_Naive=('Naive_Pred', 'sum')
).reset_index()
bu_regional['Region_Name'] = bu_regional['Region'].map(REGION_NAMES)

bu_national = mine_proj_adj.groupby('ForecastYear').agg(
    BU_Pred=('Pred_adj', 'sum'),
    BU_Naive=('Naive_Pred', 'sum')
).reset_index()
bu_national['Region'] = 'NAC'

print("Bottom-up nacional (kt):")
print(bu_national[['ForecastYear', 'BU_Pred', 'BU_Naive']].round(1).to_string(index=False))

# ---------------------------------------------------------------------------
# 9.5  Merge Direct-Regional vs Bottom-Up for comparison
# ---------------------------------------------------------------------------
direct_nat = nat_proj[['ForecastYear', 'Pred', 'Naive_Pred']].copy()
direct_nat.columns = ['ForecastYear', 'Direct_Pred', 'Naive_Pred']

comparison = direct_nat.merge(bu_national[['ForecastYear', 'BU_Pred']], on='ForecastYear')
comparison['Diff_kt'] = (comparison['Direct_Pred'] - comparison['BU_Pred']).round(1)
comparison['Diff_pct'] = ((comparison['Direct_Pred'] / comparison['BU_Pred'] - 1) * 100).round(2)

print("\n=" * 70)
print("COMPARACIÓN: Proyección Directa-Regional vs Bottom-Up (Nacional, kt)")
print("=" * 70)
print(f"{'Year':>6} | {'Direct_Reg':>12} | {'Bottom_Up':>10} | {'Naive':>10} | {'Diff_kt':>8} | {'Diff%':>7}")
print("-" * 70)
for _, row in comparison.iterrows():
    print(f"{int(row['ForecastYear']):>6} | {row['Direct_Pred']:>12.1f} | "
          f"{row['BU_Pred']:>10.1f} | {row['Naive_Pred']:>10.1f} | "
          f"{row['Diff_kt']:>8.1f} | {row['Diff_pct']:>6.1f}%")

# ---------------------------------------------------------------------------
# 9.6  Regional comparison table
# ---------------------------------------------------------------------------
direct_reg = proj_df.groupby(['Region', 'ForecastYear'])['Pred'].sum().reset_index()
direct_reg.columns = ['Region', 'ForecastYear', 'Direct_Pred']

reg_comparison = direct_reg.merge(
    bu_regional[['Region', 'ForecastYear', 'BU_Pred']], on=['Region', 'ForecastYear'], how='outer'
).fillna(0.0)

years_show = [2026, 2028, 2030, 2032]
print("\n--- Comparación por Región (años seleccionados, kt) ---")
for reg in REGION_ORDER:
    r_d = direct_reg[direct_reg['Region'] == reg].set_index('ForecastYear')['Direct_Pred']
    r_b = bu_regional[bu_regional['Region'] == reg].set_index('ForecastYear')['BU_Pred']
    parts = []
    for y in years_show:
        d = r_d.get(y, 0.0)
        b = r_b.get(y, 0.0)
        parts.append(f"{y}: D={d:.0f}/BU={b:.0f}")
    print(f"  {reg:4s} ({REGION_NAMES.get(reg,'?'):20s}): {' | '.join(parts)}")

Bottom-up nacional (kt):
 ForecastYear  BU_Pred  BU_Naive
         2026   5454.1    5471.8
         2027   5327.0    5471.8
         2028   5203.2    5471.8
         2029   5137.6    5471.8
         2030   5043.0    5471.8
         2031   4931.6    5471.8
         2032   4870.5    5471.8

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
COMPARACIÓN: Proyección Directa-Regional vs Bottom-Up (Nacional, kt)
  Year |   Direct_Reg |  Bottom_Up |      Naive |  Diff_kt |   Diff%
----------------------------------------------------------------------
  2026 |       5571.5 |     5454.1 |     5471.8 |    117.4 |    2.1%
  2027 |       5573.6 |     5327.0 |     5471.8 |    246.5 |    4.6%
  2028 |       5582.3 |     5203.2 |     5471.8 |    379.0 |    7.3%
  2029 |       5468.0 |     5137.6 |     5471.8 |    330.4 |    6.4%
  2030 |       5317.4 |     5043.0 |     5471.8 |    274.4 |    5.4%
  2031 |       5

## Sección 10 — Guardar Outputs

In [15]:
# ---------------------------------------------------------------------------
# 10.1  regional_projections_2026_2032.csv
# ---------------------------------------------------------------------------
regional_proj_out = proj_df[[
    'Region', 'Region_Name', 'ForecastYear', 'Horizonte_feat',
    'Pred', 'Naive_Pred', 'Lower', 'Upper', 'Origin_Prod', 'Region_Size'
]].copy()
regional_proj_out = regional_proj_out.rename(columns={'Horizonte_feat': 'Horizonte'})
regional_proj_out = regional_proj_out.sort_values(['Region', 'ForecastYear'])
regional_proj_out = regional_proj_out.round(3)

path_reg = f"{OUT}/regional_projections_2026_2032.csv"
regional_proj_out.to_csv(path_reg, index=False)
print(f"Guardado: {path_reg}  ({len(regional_proj_out)} filas)")

# ---------------------------------------------------------------------------
# 10.2  national_projections_2026_2032.csv
#        Include both direct-regional and bottom-up columns
# ---------------------------------------------------------------------------
nat_out = nat_proj[['ForecastYear', 'Horizonte', 'Pred', 'Naive_Pred', 'Lower', 'Upper', 'Origin_Prod']].copy()
nat_out = nat_out.merge(
    bu_national[['ForecastYear', 'BU_Pred']],
    on='ForecastYear', how='left'
)
nat_out.columns = ['ForecastYear', 'Horizonte', 'Direct_Pred', 'Naive_Pred',
                   'Direct_Lower', 'Direct_Upper', 'Origin_Prod', 'BU_Pred']
nat_out['Region']      = 'NAC'
nat_out['Region_Name'] = 'Nacional'
nat_out = nat_out.round(3)

path_nat = f"{OUT}/national_projections_2026_2032.csv"
nat_out.to_csv(path_nat, index=False)
print(f"Guardado: {path_nat}  ({len(nat_out)} filas)")

# ---------------------------------------------------------------------------
# 10.3  regional_scoreboard.csv
# ---------------------------------------------------------------------------
path_scb = f"{OUT}/regional_scoreboard.csv"
scoreboard.round(4).to_csv(path_scb, index=False)
print(f"Guardado: {path_scb}  ({len(scoreboard)} filas)")

# ---------------------------------------------------------------------------
# 10.4  regional_predictions_validation.csv
# ---------------------------------------------------------------------------
val_out_cols = [
    'Region', 'Origin_Year', 'Target_Year', 'Horizonte_feat',
    'Origin_Prod', 'Prod_Target', 'LogRatio',
    'XGB_pred', 'LGB_pred', 'Ens_pred', 'Naive_pred',
    'XGB_err', 'LGB_err', 'Ens_err', 'Naive_err'
]
path_val = f"{OUT}/regional_predictions_validation.csv"
val_all[val_out_cols].round(6).to_csv(path_val, index=False)
print(f"Guardado: {path_val}  ({len(val_all)} filas)")

# ---------------------------------------------------------------------------
# 10.5  optuna_params_regional.json
# ---------------------------------------------------------------------------
optuna_params = {
    'XGB_MultiH': best_xgb_params,
    'LGB_MultiH': best_lgb_params,
    'best_model': best_model_name,
    'n_trials':   N_TRIALS,
    'features':   FEATURE_COLS,
    'origins':    ORIGINS,
    'horizons':   HORIZONS,
}
path_params = f"{OUT}/optuna_params_regional.json"
with open(path_params, 'w') as f:
    json.dump(optuna_params, f, indent=2)
print(f"Guardado: {path_params}")

# ---------------------------------------------------------------------------
# Resumen final
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("RESUMEN FINAL")
print("=" * 60)
print(f"Mejor modelo: {best_model_name}")
nat_best = scoreboard[(scoreboard['Region'] == 'NAC') & (scoreboard['Model'] == best_model_name)].iloc[0]
print(f"  WR Nacional      : {nat_best['WR_pct']}%")
print(f"  Skill% Nacional  : {nat_best['Skill']}%")
print(f"  MASE Nacional    : {nat_best['MASE']}")
print(f"\nProducción 2025  : {prod_2025_total:.1f} kt")
print("Proyecciones directas nacionales:")
for _, row in nat_out.iterrows():
    print(f"  {int(row['ForecastYear'])}: {row['Direct_Pred']:.1f} kt "
          f"[{row['Direct_Lower']:.1f}, {row['Direct_Upper']:.1f}]  "
          f"(BU: {row['BU_Pred']:.1f} kt)")
print("\nArchivos guardados en:", OUT)

Guardado: /Users/mac/TrabajoTesis/FinalResultsFolder/03_Forecasting/regional_model/outputs/regional_projections_2026_2032.csv  (56 filas)
Guardado: /Users/mac/TrabajoTesis/FinalResultsFolder/03_Forecasting/regional_model/outputs/national_projections_2026_2032.csv  (7 filas)
Guardado: /Users/mac/TrabajoTesis/FinalResultsFolder/03_Forecasting/regional_model/outputs/regional_scoreboard.csv  (27 filas)
Guardado: /Users/mac/TrabajoTesis/FinalResultsFolder/03_Forecasting/regional_model/outputs/regional_predictions_validation.csv  (407 filas)
Guardado: /Users/mac/TrabajoTesis/FinalResultsFolder/03_Forecasting/regional_model/outputs/optuna_params_regional.json

RESUMEN FINAL
Mejor modelo: LGB
  WR Nacional      : 51.8%
  Skill% Nacional  : -0.16%
  MASE Nacional    : 1.0016

Producción 2025  : 5471.8 kt
Proyecciones directas nacionales:
  2026: 5571.5 kt [5338.2, 6073.8]  (BU: 5454.1 kt)
  2027: 5573.6 kt [5289.6, 6088.7]  (BU: 5327.0 kt)
  2028: 5582.3 kt [5307.6, 6107.2]  (BU: 5203.2 kt)
  2